In [0]:
spark.conf.set(
    "fs.azure.account.key.adlsstoragedelta.dfs.core.windows.net",
    dbutils.secrets.get(scope="deltaapp", key="scopeKey"))

In [0]:
display(dbutils.secrets.listScopes())

In [0]:
display(dbutils.secrets.list("deltaapp"))

In [0]:
display(dbutils.fs.ls("abfss://delta@adlsstoragedelta.dfs.core.windows.net/"))


In [0]:
training_df = spark.read.csv("abfss://delta@adlsstoragedelta.dfs.core.windows.net/Training1.csv", header=True, inferSchema=True)


In [0]:
display(training_df.limit(5))

In [0]:
delta_path = "abfss://delta@adlsstoragedelta.dfs.core.windows.net/training_data/"

In [0]:
#Creating delta table using external storage
training_df.write.format("delta").mode("overwrite").save(delta_path)

In [0]:
%sql
--Creating delta table on top of delta lake
select * from delta.`abfss://delta@adlsstoragedelta.dfs.core.windows.net/training_data/`;

In [0]:
%sql
update delta.`abfss://delta@adlsstoragedelta.dfs.core.windows.net/training_data/` set cost = 2300 where Employee_ID = 4;

In [0]:
%sql
select * from delta.`abfss://delta@adlsstoragedelta.dfs.core.windows.net/training_data/` where Employee_ID = 4;

In [0]:
%sql
--INSERT
insert into delta.`abfss://delta@adlsstoragedelta.dfs.core.windows.net/training_data/` values('2022-01-01', 100, 'Python', 5, 2000, 'LinkedIn');


In [0]:
%sql
select * from delta.`abfss://delta@adlsstoragedelta.dfs.core.windows.net/training_data/` where Employee_ID = 5;

In [0]:
%sql
--DELETE
delete from delta.`abfss://delta@adlsstoragedelta.dfs.core.windows.net/training_data/` where Employee_ID = 5 and Supplier = "LinkedIn";

In [0]:
%sql
select * from delta.`abfss://delta@adlsstoragedelta.dfs.core.windows.net/training_data/` where Employee_ID = 5;

### Time Travel

In [0]:
%sql
describe history delta.`abfss://delta@adlsstoragedelta.dfs.core.windows.net/training_data/`;

In [0]:
%sql
select * from delta.`abfss://delta@adlsstoragedelta.dfs.core.windows.net/training_data/` version as of 2;

In [0]:
%sql
--Reverting back to previous version that is before delete operation
RESTORE TABLE delta.`abfss://delta@adlsstoragedelta.dfs.core.windows.net/training_data/` TO VERSION AS OF 2;

In [0]:
%sql
select * from delta.`abfss://delta@adlsstoragedelta.dfs.core.windows.net/training_data/` where Employee_ID = 5;

In [0]:
%sql
describe history delta.`abfss://delta@adlsstoragedelta.dfs.core.windows.net/training_data/`;

Manually Checking with Unity Catalog Table

In [0]:
%sql
select * from deltadbx_7405605814788338.deltaschema.emp_1;

Deleting a Catalog

In [0]:
%sql
DROP CATALOG deltadbx_7405605814788338 CASCADE;


In [0]:
%sql
DROP CATALOG deltaCatalog CASCADE;

### Additional Informations

In [0]:
%sql
--Data Recovery  default is 7 days and we can increase upto 30 days using below query
alter table delta.`abfss://delta@adlsstoragedelta.dfs.core.windows.net/training_data/` /
set tblproperties (delta.logRetentionDuration = '30 days', delta.deletedFileRetentionDuration = '30 days);
    